In [3]:

import json
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
sys.path.append(str(Path().resolve().parent))
from src.preprocessing import build_pipeline
from src.feature_engineering import add_engineered_features


In [4]:

# Locate project root
if Path("data/train.csv").exists():
    PROJECT_ROOT = Path(".")
elif Path("../data/train.csv").exists():
    PROJECT_ROOT = Path("..")
else:
    raise FileNotFoundError("Could not locate project data directory.")

DATA_DIR = PROJECT_ROOT / "data"

# Load original datasets
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Sample submission:", sample_submission.shape)


Train: (1460, 81)
Test: (1459, 80)
Sample submission: (1459, 2)


In [5]:

# Preserve identifiers
train_ids = train_df["Id"].copy()
test_ids = test_df["Id"].copy()

# Training predictors and target
X_full = train_df.drop(
    columns=["Id", "SalePrice"]
)

y_full = train_df["SalePrice"]

# Test predictors
X_test = test_df.drop(columns=["Id"])

# Ensure test columns match training columns
assert set(X_full.columns) == set(X_test.columns)

X_test = X_test[X_full.columns]

print("X_full:", X_full.shape)
print("y_full:", y_full.shape)
print("X_test:", X_test.shape)

assert X_full.isna().sum().sum() >= 0
assert y_full.isna().sum() == 0
assert train_ids.is_unique
assert test_ids.is_unique


X_full: (1460, 79)
y_full: (1460,)
X_test: (1459, 79)


In [6]:

# Apply the same feature engineering to training and test data
X_full_fe = add_engineered_features(
    X_full,
    include_interactions=True
)

X_test_fe = add_engineered_features(
    X_test,
    include_interactions=True
)

# Verify feature consistency
assert list(X_full_fe.columns) == list(X_test_fe.columns)

print("Training features:", X_full_fe.shape)
print("Test features:", X_test_fe.shape)

print("\nNew engineered features:")
print(
    sorted(
        set(X_full_fe.columns) - set(X_full.columns)
    )
)


Training features: (1460, 86)
Test features: (1459, 86)

New engineered features:
['HouseAgeAtSale', 'QualityTotalAreaInteraction', 'TotalAreaIncludingBasement', 'TotalBathrooms', 'TotalFinishedBsmtSF', 'TotalOutdoorArea', 'YearsSinceRemodel']


In [7]:

config_path = PROJECT_ROOT / "config" / "phase7_best_params.json"

with open(config_path, "r", encoding="utf-8") as file:
    best_model_config = json.load(file)

# Extract actual estimator parameter names
svr_params = {
    key.rsplit("__", 1)[-1]: value
    for key, value in best_model_config[
        "svr_best_params"
    ].items()
}

gb_params = {
    key.rsplit("__", 1)[-1]: value
    for key, value in best_model_config[
        "gradient_boosting_best_params"
    ].items()
}

ensemble_weights = best_model_config["ensemble_weights"]

print("SVR parameters:")
print(svr_params)

print("\nGradient Boosting parameters:")
print(gb_params)

print("\nEnsemble weights:")
print(ensemble_weights)


SVR parameters:
{'C': 5.4182823195332395, 'epsilon': 0.003124565071260871, 'gamma': 0.001095266274863256}

Gradient Boosting parameters:
{'learning_rate': 0.04597505784732168, 'max_depth': 3, 'min_samples_leaf': 6, 'n_estimators': 589, 'subsample': 0.6831766651472755}

Ensemble weights:
{'SVR': 2, 'Gradient Boosting': 1}


In [8]:

def make_final_pipeline(estimator, X_reference, scale_numeric):

    preprocessing_pipeline = build_pipeline(
        X_reference,
        estimator,
        scale_numeric=scale_numeric
    )

    return TransformedTargetRegressor(
        regressor=preprocessing_pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )


# Reconstruct SVR
svr = SVR(
    kernel="rbf",
    **svr_params
)

# Reconstruct Gradient Boosting
gradient_boosting = GradientBoostingRegressor(
    random_state=42,
    **gb_params
)

# Build independent preprocessing pipelines
svr_final = make_final_pipeline(
    svr,
    X_full_fe,
    scale_numeric=True
)

gb_final = make_final_pipeline(
    gradient_boosting,
    X_full_fe,
    scale_numeric=False
)


In [9]:

print("Training final SVR...")
svr_final.fit(X_full_fe, y_full)

print("Training final Gradient Boosting...")
gb_final.fit(X_full_fe, y_full)

print("Both models have been trained successfully.")


Training final SVR...
Training final Gradient Boosting...
Both models have been trained successfully.


In [10]:

svr_predictions = svr_final.predict(X_test_fe)

gb_predictions = gb_final.predict(X_test_fe)

predictions_df = pd.DataFrame({
    "Id": test_ids,
    "SVR_prediction": svr_predictions,
    "GB_prediction": gb_predictions
})

display(predictions_df.head())


,Id,SVR_prediction,GB_prediction
0,1461,119895.361266,124991.076422
1,1462,164486.374159,160744.223930
2,1463,191367.827787,175959.070016
3,1464,201107.156993,195502.907316
4,1465,183405.759870,181381.843968


In [11]:

svr_weight = ensemble_weights["SVR"]
gb_weight = ensemble_weights["Gradient Boosting"]

ensemble_predictions = (
    svr_weight * svr_predictions +
    gb_weight * gb_predictions
) / (svr_weight + gb_weight)

# Prices cannot be negative
ensemble_predictions = np.clip(
    ensemble_predictions,
    0,
    None
)

print("Number of predictions:", len(ensemble_predictions))
print("Minimum predicted price:", ensemble_predictions.min())
print("Maximum predicted price:", ensemble_predictions.max())
print("Mean predicted price:", ensemble_predictions.mean())


Number of predictions: 1459
Minimum predicted price: 49070.741288327175
Maximum predicted price: 539404.4262161764
Mean predicted price: 178884.4606104025


In [12]:

# Verify test IDs match sample submission
assert np.array_equal(
    test_ids.to_numpy(),
    sample_submission["Id"].to_numpy()
)

submission = sample_submission.copy()

submission["SalePrice"] = ensemble_predictions

# Validate final submission
assert len(submission) == len(test_df)
assert submission.columns.tolist() == ["Id", "SalePrice"]
assert submission["Id"].is_unique
assert submission["SalePrice"].notna().all()
assert np.isfinite(submission["SalePrice"]).all()
assert (submission["SalePrice"] >= 0).all()

# Save without DataFrame index
submission_dir = PROJECT_ROOT / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

submission_path = (
    submission_dir /
    "submission_phase8_ensemble_svr_gb.csv"
)

submission.to_csv(submission_path, index=False)

print("Submission saved:", submission_path)
display(submission.head(10))


Submission saved: ..\submissions\submission_phase8_ensemble_svr_gb.csv


,Id,SalePrice
0,1461,121593.932985
1,1462,163238.990749
2,1463,186231.575197
3,1464,199239.073767
4,1465,182731.121236
5,1466,174764.864987
6,1467,179674.055011
7,1468,166579.285096
8,1469,190212.792181
9,1470,127448.001089


In [13]:

# Read the saved submission again
submission_check = pd.read_csv(submission_path)

print("Submission shape:", submission_check.shape)
print("\nData types:")
print(submission_check.dtypes)

print("\nMissing values:")
print(submission_check.isna().sum())

print("\nPrice summary:")
display(submission_check["SalePrice"].describe())

assert submission_check.shape == (1459, 2)
assert submission_check["SalePrice"].notna().all()
assert np.isfinite(submission_check["SalePrice"]).all()
assert (submission_check["SalePrice"] >= 0).all()

print("\nSubmission validation passed.")


Submission shape: (1459, 2)

Data types:
Id             int64
SalePrice    float64
dtype: object

Missing values:
Id           0
SalePrice    0
dtype: int64

Price summary:


count      1459.000000
mean     178884.460610
std       76379.074853
min       49070.741288
25%      128385.508308
50%      158680.527064
75%      210308.655480
max      539404.426216
Name: SalePrice, dtype: float64


Submission validation passed.



## Final Kaggle Evaluation

The final SVR + Gradient Boosting ensemble achieved
a Kaggle RMSLE of **0.11752**.

This improves upon the previous SVR submission,
which achieved 0.11943.

The relative RMSLE reduction was approximately 1.60%.

The final ensemble was trained using all 1,460 labeled
observations and generated predictions for the 1,459
Kaggle test observations.

## Final conclusion

Feature Engineering and model ensembling improved
the predictive performance of the original SVR model.

The combination of optimized SVR and Gradient Boosting
with weights [2, 1] achieved the best Kaggle result
among the evaluated approaches.

The project demonstrates a complete supervised
machine learning workflow, including data exploration,
preprocessing, feature engineering, model selection,
hyperparameter optimization, ensembling and final evaluation.
